In [ ]:
from pathlib import Path

import duckdb

from thesis.config import load_config
from thesis.vlm.run import configured_clients

ablation_dir = Path("/Users/phillip/repositories/bachelor-thesis/data/ablation")

con = duckdb.connect(str(ablation_dir / "manifest.duckdb"), read_only=True)
red_crosshair = con.sql(
    "SELECT * FROM samples WHERE marker_shape = ? AND marker_color = ?",
    params=["crosshair", str((230, 20, 20))],
).df()

client = next(c for c in configured_clients() if c.provider == "anthropic")
prompt_text = next(p["text"] for p in load_config("prompts")["prompts"] if p["id"] == "minimal")

for row in red_crosshair[:10].itertuples():
    image_bytes = (ablation_dir / row.image_path).read_bytes()
    result = client.classify(image_bytes, prompt_text)
    print(row.sample_id, "->", result.vlm_response.raw_text)

ee8390b3eb091275b064d31c9c4efbcf5f9200bb2c876141677b5a5195e429df -> Yes
165a679940c5f5fb4324fb416c62e310b7322f1937a902e46f94e69246743f1b -> yes
230d2257e28683bc4077a05b25b8b03aec0940ea67ed6a0ba9f9fd10d30a3524 -> yes
d5ff7f5a1bb32660cf4553c4f4838d16bdbb0e1571f82897631e7ee3872b3cf5 -> no
becf9efdf914eb436b7e63636abdc95ffcc5dbe89240d71a8e46c1190d985112 -> No
93c42c33c75dcdb8bdd55fe0d7177b00a6627c1aa4d38e66f3bbbe18dad0f2d2 -> no
f6ad2abadd3315f17c0a1804da606fcdd9f9eadab88a71d5eb8051ad0ca2c144 -> no
933134468f91785fdaac5f43feaa2d689dd725e765a2457a3ff4dd6afe19d98c -> no
493f60a0e6d99c0ff54c9c57045249c66ba1fba5aaea02a0488e4fa0997a1dce -> no
bbcaa86341daa40ef2a3614591b794ed367f83bdd82d0980d3ef490f42ae66af -> Yes


In [5]:
len(red_crosshair)

1012

In [8]:
# length of entries inside dir
dir = Path("/Users/phillip/repositories/bachelor-thesis/data/vlm_cache/anthropic")
entries = list(dir.iterdir())
len(entries)

2851

In [7]:
from dotenv import load_dotenv

from thesis.vlm.run import run_dataset

load_dotenv()

ablation_dir = Path("/Users/phillip/repositories/bachelor-thesis/data/ablation")

image_bytes_by_row = [(ablation_dir / str(row.image_path)).read_bytes() for _, row in red_crosshair.iterrows()]

all_results = {}
for prompt in load_config("prompts")["prompts"]:
    items = [(image_bytes, prompt["text"]) for image_bytes in image_bytes_by_row]
    all_results[prompt["id"]] = run_dataset(items, clients=configured_clients())

KeyboardInterrupt: 